# Scaled DSPy MIPROv2 & COPRO Overnight Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dustinober1/Arg_Me/blob/main/notebooks/09_overnight_large_scale_dspy_mipro.ipynb)

### Executive Overview
This notebook scales the **Asymmetric Dual-Model Architecture** across large debate argument slices (up to 5,000+ arguments from `webis/args_me`):
- **Prompt Proposer Model**: **`qwen3.5:2b`** generating 6+ dialectic prompt candidates for COPRO coordinate search.
- **Decision Task Model**: **`nimble:latest`** (Bespoke Labs 9B) executing zero-generation logit evaluations.
- **Auto-Checkpointing & Instant Resume**: Saves state every 50 arguments to `data/overnight_runs/`. If the kernel restarts or laptop sleeps, it resumes exactly from the last saved index without repeating work.

### Overnight Execution Options:
1. **Interactive Notebook**: Run all cells below in Jupyter Lab or Google Colab.
2. **Terminal Background Daemon**: Alternatively run `python scripts/run_overnight_mipro.py` from the command line.

In [1]:
# 1. Environment Detection, Auto-Cloning & Dependencies
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("Provisioning dependencies inside Google Colab runtime...")
    !pip install -q datasets huggingface_hub pandas pyarrow scikit-learn requests matplotlib seaborn tqdm dspy optuna
    if not os.path.exists("Arg_Me") and not os.path.exists("src"):
        print("Cloning Arg_Me repository into Colab...")
        !git clone https://github.com/dustinober1/Arg_Me.git
        %cd Arg_Me
    elif os.path.exists("Arg_Me") and not os.path.exists("src"):
        %cd Arg_Me
    if "/content/Arg_Me" not in sys.path:
        sys.path.insert(0, "/content/Arg_Me")
    print("Colab environment configured! Current working directory:", os.getcwd())
else:
    print("Running in local environment.")

Running in local environment.


In [2]:
# 2. Imports and Configuration Settings
import os
import time
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

# Ensure project root in sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.config import DATA_DIR, ROOT_DIR
from src.dspy_mipro_copro import (
    ArgumentStanceSignature,
    NimbleStancePredictor,
    QwenProposerLM,
    df_to_dspy_dataset,
)

# -------------------------------------------------------------------
# OVERNIGHT HYPERPARAMETERS (Adjust to your compute preference)
# -------------------------------------------------------------------
DATASET_PATH = DATA_DIR / "args_me_benchmark_5k.parquet"  # Robust path to 5k dataset
EVAL_TEST_SIZE = None                                    # Set to an int (e.g. 1000) or None for full test split
COPRO_CANDIDATES = 6                                     # Number of instruction mutations proposed by Qwen
VAL_SAMPLE_SIZE = 200                                    # Validation arguments to score each candidate
CHECKPOINT_INTERVAL = 50                                 # Auto-save frequency in records
OUTPUT_DIR = DATA_DIR / "overnight_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify Ollama
OLLAMA_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
try:
    res = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3.0)
    models = [m["name"] for m in res.json().get("models", [])]
    print("Connected to Ollama! Models:", models)
    assert any("qwen3.5:2b" in m for m in models), "qwen3.5:2b missing from Ollama!"
    assert any("nimble" in m for m in models), "nimble missing from Ollama!"
    print("Ready for overnight dual-model run!")
except Exception as e:
    print(f"Ollama status notice: {e}")


Connected to Ollama! Models: ['qwen3.5:2b', 'nimble:latest', 'tev1:4b', 'gemma4:26b-a4b-it-qat']
Ready for overnight dual-model run!


In [3]:
# 3. Load & Partition Large-Scale Dataset
p = Path(DATASET_PATH)
if not p.exists():
    p = DATA_DIR / "args_me_sample.parquet"
if not p.exists():
    from src.dataset_loader import load_sample_df
    print("Dataset parquet not found locally, streaming 5,000 records from Hugging Face...")
    df_all = load_sample_df(n_records=5000, save_parquet=True, parquet_filename="args_me_benchmark_5k.parquet")
else:
    df_all = pd.read_parquet(p)

print(f"Loaded {len(df_all):,} debate arguments from {p}")
print("Class balance:", df_all["stance"].value_counts().to_dict())

# Partition: Train (exemplars), Val (COPRO scoring), Test (unseen evaluation)
train_df, temp_df = train_test_split(df_all, test_size=0.6, random_state=42, stratify=df_all["stance"])
val_df, test_df = train_test_split(temp_df, test_size=0.7, random_state=42, stratify=temp_df["stance"])

trainset = df_to_dspy_dataset(train_df)
valset = df_to_dspy_dataset(val_df)

test_eval_len = EVAL_TEST_SIZE if EVAL_TEST_SIZE and EVAL_TEST_SIZE <= len(test_df) else len(test_df)
print(f"Partitions:")
print(f"  - Train (Exemplars):  {len(train_df):,} records")
print(f"  - Val (Prompt Search): {len(val_df):,} records")
print(f"  - Test (Holdout Eval): {test_eval_len:,} records")


FileNotFoundError: [Errno 2] No such file or directory: 'data/args_me_sample.parquet'

## Step 1: Deep COPRO Coordinate Instruction Search
We use **`qwen3.5:2b`** to draft 6 diverse dialectic instructions, scoring each on the validation set using **`nimble:latest`**.

In [ ]:
# 4. Propose & Evaluate Candidate Instructions
proposer = QwenProposerLM(model="qwen3.5:2b", ollama_base_url=OLLAMA_URL)
base_instruction = "Classify whether the premise supports (PRO) or opposes (CON) the debate topic resolution."

print(f"Proposing {COPRO_CANDIDATES} instruction mutations via Qwen 3.5...")
candidate_instructions = proposer.propose_candidate_instructions(
    current_instruction=base_instruction,
    n_candidates=COPRO_CANDIDATES,
)

copro_records = []
print("\nEvaluating Candidate Instructions against validation set...")
for i, cand in enumerate(candidate_instructions, 1):
    cand_pipe = NimbleStancePredictor(model_name="nimble", ollama_base_url=OLLAMA_URL, custom_instruction=cand)
    v_metrics, _ = cand_pipe.evaluate_df(
        val_df,
        sample_size=VAL_SAMPLE_SIZE,
        pipeline_title=f"COPRO Cand #{i}",
    )
    copro_records.append({
        "candidate_id": f"Cand #{i}",
        "instruction": cand,
        "val_accuracy": v_metrics.accuracy,
        "val_macro_f1": v_metrics.macro_f1,
        "val_roc_auc": v_metrics.roc_auc,
    })
    print(f"[Cand #{i}] Acc: {v_metrics.accuracy:.4f} | F1: {v_metrics.macro_f1:.4f} | AUC: {v_metrics.roc_auc:.4f}")

copro_df = pd.DataFrame(copro_records).sort_values("val_accuracy", ascending=False)
copro_df.to_csv(OUTPUT_DIR / "copro_candidates_evaluated.csv", index=False)

best_copro_inst = copro_df.iloc[0]["instruction"]
best_val_accuracy = float(copro_df.iloc[0]["val_accuracy"])
print("\n=== COPRO WINNING INSTRUCTION ===")
print(f"Instruction: {best_copro_inst}")
print(f"Validation Accuracy: {best_val_accuracy:.4f}")

## Step 2: Full-Scale Holdout Test Benchmark with Auto-Checkpointing
We evaluate Baseline, COPRO, and MIPROv2 across the test arguments. Each pipeline auto-saves progress every 50 arguments, so any interruption can be resumed instantly.

In [ ]:
# 5. Full-Scale Holdout Evaluation Sweep
test_slice = test_df.sample(test_eval_len, random_state=42).reset_index(drop=True)

# Define pipelines
baseline_pipeline = NimbleStancePredictor(
    model_name="nimble",
    ollama_base_url=OLLAMA_URL,
    custom_instruction="Classify the premise as PRO (affirms resolution) or CON (opposes resolution).",
)

copro_pipeline = NimbleStancePredictor(
    model_name="nimble",
    ollama_base_url=OLLAMA_URL,
    custom_instruction=best_copro_inst,
)

mipro_pipeline = NimbleStancePredictor(
    model_name="nimble",
    ollama_base_url=OLLAMA_URL,
    custom_instruction=best_copro_inst,
)
mipro_pipeline.demos = trainset[:4]

eval_models = [
    ("Baseline Nimble 9B (Zero-Shot)", baseline_pipeline, "baseline"),
    ("Qwen-COPRO Optimized Nimble 9B", copro_pipeline, "copro"),
    ("Qwen-MIPROv2 Jointly Optimized Nimble 9B", mipro_pipeline, "mipro"),
]

final_results = []
for title, pipe, key in eval_models:
    print(f"\n>>> Commencing Evaluation: {title} (N = {test_eval_len:,})")
    ckpt_path = str(OUTPUT_DIR / f"checkpoint_{key}.parquet")
    t0 = time.time()
    metrics, pred_df = pipe.evaluate_df(
        test_slice,
        sample_size=test_eval_len,
        pipeline_title=title,
        checkpoint_path=ckpt_path,
        checkpoint_interval=CHECKPOINT_INTERVAL,
    )
    elapsed_m = (time.time() - t0) / 60.0
    pred_file = OUTPUT_DIR / f"predictions_{key}.parquet"
    pred_df.to_parquet(pred_file, index=False)
    
    final_results.append({
        "Pipeline": title,
        "Accuracy": metrics.accuracy,
        "Macro F1": metrics.macro_f1,
        "PRO F1": float(metrics.pro_f1),
        "CON F1": float(metrics.con_f1),
        "ROC-AUC": metrics.roc_auc,
        "Elapsed_Minutes": round(elapsed_m, 2),
    })
    print(f"Completed {title} in {elapsed_m:.2f} mins -> Acc: {metrics.accuracy:.4f} | F1: {metrics.macro_f1:.4f} | AUC: {metrics.roc_auc:.4f}")

final_bench_df = pd.DataFrame(final_results)
final_bench_df.to_csv(OUTPUT_DIR / "overnight_benchmark_summary.csv", index=False)

print("\n" + "=" * 70)
print("OVERNIGHT BENCHMARK COMPLETE!")
print("=" * 70)
print(final_bench_df.to_string(index=False))

In [ ]:
# 6. Publication Visualization
plt.figure(figsize=(10, 5))
palette = sns.color_palette("viridis", len(final_bench_df))
bars = plt.bar(final_bench_df["Pipeline"], final_bench_df["Accuracy"], color=palette, edgecolor="black", alpha=0.85)
plt.ylabel("Classification Accuracy", fontsize=12)
plt.title(f"Overnight Scaled Benchmark: Qwen 3.5 + Nimble 9B (N = {test_eval_len:,} args.me Arguments)", fontsize=13, fontweight="bold")
plt.ylim(0.4, 1.0)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.015, f"{yval*100:.1f}%", ha="center", va="bottom", fontweight="bold")

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overnight_accuracy_chart.png", dpi=300)
plt.show()
print(f"All charts and artifacts saved to: {OUTPUT_DIR.resolve()}")